In [1]:
from unsloth import FastModel
import torch

model, tokenizer = FastModel.from_pretrained(
    model_name = "wizardoftrap/LFM2.5-1.2B-hi-it",
    dtype = None,
    max_seq_length = 2048,
    load_in_4bit = False,
    full_finetuning = False,
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.1.3: Fast Lfm2 patching. Transformers: 4.57.3.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.034 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.6.0+cu124. CUDA: 8.9. CUDA Toolkit: 12.4. Triton: 3.2.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.29.post3. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors:   0%|          | 0.00/2.34G [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/434 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [2]:
model = FastModel.get_peft_model(
    model,
    finetune_vision_layers     = False,
    finetune_language_layers   = True,
    finetune_attention_modules = True,
    finetune_mlp_modules       = True,

    r = 64,
    lora_alpha = 64,
    lora_dropout = 0,
    bias = "none",
    random_state = 3407,
)

Unsloth: Making `model.base_model.model.model` require gradients


```
<|startoftext|><|im_start|>user
Hello!<|im_end|>
<|im_start|>assistant
Hey there!<|im_end|>
```

In [3]:
tokenizer.apply_chat_template([
    {"role" : "user", "content" : "Hello!"},
    {"role" : "assistant", "content" : "Hey there!"}
], tokenize = False)

'<|startoftext|><|im_start|>user\nHello!<|im_end|>\n<|im_start|>assistant\nHey there!<|im_end|>\n'

In [4]:
from datasets import load_dataset
dataset = load_dataset("wizardoftrap/indianHistoryEnhanced",split=["train"])

README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/2.28M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/5354 [00:00<?, ? examples/s]

In [6]:
dataset=dataset[0]

In [8]:
def convert_input_output_to_messages(examples):
    messages_list = []
    for inp, out in zip(examples['input'], examples['output']):
        messages = [
            {"role": "user", "content": inp},
            {"role": "assistant", "content": out}
        ]
        messages_list.append(messages)
    return {"messages": messages_list}

dataset = dataset.map(convert_input_output_to_messages, batched=True)

Map:   0%|          | 0/5354 [00:00<?, ? examples/s]

In [10]:
dataset[0]

{'input': "How did modern nationalism in Europe relate to the formation of nation‑states, and what changes did it bring to people's sense of identity?",
 'output': 'Modern nationalism in Europe became closely linked with the creation of nation‑states, where political boundaries were drawn to match a shared cultural identity. This process altered how people understood themselves, shifting from regional or imperial affiliations to a national consciousness. New symbols such as flags, anthems, and icons were introduced to represent the emerging nation. Songs and slogans were used to foster a collective spirit and a sense of belonging among citizens. The redefinition of community boundaries encouraged people to see themselves as members of a larger national whole. Overall, the rise of nationalism transformed personal and collective identity by emphasizing common heritage, language, and destiny.',
 'messages': [{'content': "How did modern nationalism in Europe relate to the formation of nati

In [11]:
def formatting_prompts_func(examples):
    texts = []
    for messages in examples["messages"]:
        if messages:
            text = tokenizer.apply_chat_template(
                messages,
                tokenize = False,
                add_generation_prompt = False,
            )
            texts.append(text.removeprefix(tokenizer.bos_token))
        else:
            texts.append("")  # Placeholder for empty messages
    return { "text" : texts }

dataset = dataset.map(formatting_prompts_func, batched = True)

Map:   0%|          | 0/5354 [00:00<?, ? examples/s]

In [13]:
dataset[0]["text"]

"<|im_start|>user\nHow did modern nationalism in Europe relate to the formation of nation‑states, and what changes did it bring to people's sense of identity?<|im_end|>\n<|im_start|>assistant\nModern nationalism in Europe became closely linked with the creation of nation‑states, where political boundaries were drawn to match a shared cultural identity. This process altered how people understood themselves, shifting from regional or imperial affiliations to a national consciousness. New symbols such as flags, anthems, and icons were introduced to represent the emerging nation. Songs and slogans were used to foster a collective spirit and a sense of belonging among citizens. The redefinition of community boundaries encouraged people to see themselves as members of a larger national whole. Overall, the rise of nationalism transformed personal and collective identity by emphasizing common heritage, language, and destiny.<|im_end|>\n"

In [21]:
from trl import SFTTrainer, SFTConfig
trainer = SFTTrainer(
    model = model,
    tokenizer = tokenizer,
    train_dataset = dataset,
    args = SFTConfig(
        dataset_text_field = "text",
        per_device_train_batch_size = 2,
        gradient_accumulation_steps = 4,
        warmup_steps = 5,
        num_train_epochs = 5,
        learning_rate = 2e-4,
        logging_steps = 1,
        optim = "adamw_8bit",
        weight_decay = 0.01,
        lr_scheduler_type = "linear",
        seed = 3407,
        report_to = "none",
        output_dir = "./checkpoints",
    ),
)

In [22]:
tokenizer.decode(trainer.train_dataset[100]["input_ids"])

'<|startoftext|><|im_start|>user\nWhy did Mahatma Gandhi oppose separate electorates for Dalits, and what action did he take in response to the British concession?<|im_end|>\n<|im_start|>assistant\nMahatma Gandhi opposed separate electorates because he feared they would hinder the integration of Dalits into wider Indian society. He argued that voting on the basis of caste would reinforce social divisions rather than promote unity. When the British government initially accepted Ambedkar’s demand for separate electorates, Gandhi responded by beginning a fast unto death. His fast was intended to pressure the government and the Dalit leadership to reconsider the demand. Gandhi’s personal sacrifice highlighted his conviction that a common electorate was essential for national cohesion. Ultimately, his protest led to the compromise known as the Poona Pact.<|im_end|>\n'

In [23]:
gpu_stats = torch.cuda.get_device_properties(0)
start_gpu_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
max_memory = round(gpu_stats.total_memory / 1024 / 1024 / 1024, 3)
print(f"GPU = {gpu_stats.name}. Max memory = {max_memory} GB.")
print(f"{start_gpu_memory} GB of memory reserved.")

GPU = NVIDIA L4. Max memory = 22.034 GB.
2.355 GB of memory reserved.


In [24]:
trainer_stats = trainer.train()

The model is already on multiple devices. Skipping the move to device specified in `args`.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 5,354 | Num Epochs = 5 | Total steps = 3,350
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 36,569,088 of 1,206,909,696 (3.03% trained)


Step,Training Loss
1,2.671900
2,2.686900
3,2.490200
4,2.441500
5,2.371400
6,2.290100
7,2.171100
8,2.329600
9,2.112500
10,2.001600


Unsloth: Will smartly offload gradients to save VRAM!


In [25]:
used_memory = round(torch.cuda.max_memory_reserved() / 1024 / 1024 / 1024, 3)
used_memory_for_lora = round(used_memory - start_gpu_memory, 3)
used_percentage = round(used_memory / max_memory * 100, 3)
lora_percentage = round(used_memory_for_lora / max_memory * 100, 3)
print(f"{trainer_stats.metrics['train_runtime']} seconds used for training.")
print(
    f"{round(trainer_stats.metrics['train_runtime']/60, 2)} minutes used for training."
)
print(f"Peak reserved memory = {used_memory} GB.")
print(f"Peak reserved memory for training = {used_memory_for_lora} GB.")
print(f"Peak reserved memory % of max memory = {used_percentage} %.")
print(f"Peak reserved memory for training % of max memory = {lora_percentage} %.")

2002.7954 seconds used for training.
33.38 minutes used for training.
Peak reserved memory = 2.705 GB.
Peak reserved memory for training = 0.35 GB.
Peak reserved memory % of max memory = 12.276 %.
Peak reserved memory for training % of max memory = 1.588 %.


In [35]:
messages = [{
    "role": "user",
    "content": "Tell me about Indus valley civilization",
}]
inputs = tokenizer.apply_chat_template(
    messages,
    add_generation_prompt = True,
    return_tensors = "pt",
    tokenize = True,
    return_dict = True,
).to("cuda")

from transformers import TextStreamer
_ = model.generate(
    **inputs,
    max_new_tokens = 1024,
    temperature = 0.3, min_p = 0.15, repetition_penalty = 1.05,
    streamer = TextStreamer(tokenizer, skip_prompt = True),
)

The Indus valley civilization, also known 

as the Harappan or Indus civilisation, was one of the world’s oldest civilizations. It flourished in the north‑west part of the Indian subcontinent, mainly along the Indus River and its tributaries. Archaeologists have uncovered numerous sites across present‑day Pakistan and western India. The civilisation is distinguished by its sophisticated urban planning, including grid‑like streets, drainage systems, and standardized bricks. Its people crafted seals, beads, weights, and metal objects, showing advanced technological skills. They also produced a variety of artefacts such as pottery, toys, and figurines. Although the exact duration and end‑date remain uncertain, its influence is evident in later Indian cultural elements.<|im_end|>


In [28]:

model.push_to_hub_merged("wizardoftrap/LFM2.5-1.2B-his", tokenizer,token = "hf_HlU*********")

Found HuggingFace hub cache directory: /home/wizardoftrap_sp/.cache/huggingface/hub
Checking cache directory for required files...


Unsloth: Copying 1 files from cache to `wizardoftrap/LFM2.5-1.2B-his`: 100%|██████████| 1/1 [00:01<00:00,  1.57s/it]


Successfully copied all 1 files from cache to `wizardoftrap/LFM2.5-1.2B-his`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

Unsloth: Merging weights into 16bit: 100%|██████████| 1/1 [00:38<00:00, 38.00s/it]


Unsloth: Merge process complete. Saved to `/home/wizardoftrap_sp/lfm-sft/wizardoftrap/LFM2.5-1.2B-his`
